# CinderTrace clean rebuild — Stage 3: verified downloads and daily observation accounting

Stage 2 passed for **20261114T163518Z_5b1b1c54**: 60 daily L3 files, two Sri Lanka tiles, approximately 37.09 MiB, and a verified first science file. Run this notebook using the same Google Drive account and a CPU runtime. It works with a Colab runtime from VS Code or browser Colab; no browser upload widget is used.

This stage downloads the remaining files, verifies them, resolves identical daily overlaps, and produces separate Terra/Aqua fire-mask and observation-support summaries. It does not alter the frontend, restore archive zeros, fit transfer models, fill sensor gaps, or publish datasets. L3 grid-pixel-days are not production H3 cell-days or raw detection counts.

Run cells in order. A failed check is a checkpoint requiring review, not a reason to replace missing values with zero.


## 1. Install the same recorded environment

If Colab requests a runtime restart, restart and then rerun the setup cells.


In [12]:
%pip -q install earthaccess==0.19.0 requests==2.32.4 pandas==2.2.3 numpy==2.1.3 pyarrow==23.0.1 duckdb==1.3.2 shapely==2.2.0 h3==4.5.0 pyhdf==0.11.7 pyproj==3.8.0 scipy==1.16.3


## 2. Resume the run and verify the Stage 2 inputs


In [13]:
from google.colab import drive
from pathlib import Path
import json, hashlib, re, urllib.parse
from datetime import datetime, timezone
import importlib.metadata as metadata
import pandas as pd
from shapely.geometry import shape, Polygon

drive.mount('/content/drive')
RUN_ID = '20261114T163518Z_5b1b1c54'
RUN = Path('/content/drive/MyDrive/cindertrace_clean_rebuild/runs') / RUN_ID
assert RUN.is_dir(), 'The Stage 1 run is not present in this Google Drive account.'
CONFIG = json.loads((RUN/'config.json').read_text())
COLLECTIONS = json.loads((RUN/'inventory/collections.json').read_text())
QUERY_LOG = json.loads((RUN/'inventory/query_log.json').read_text())
expected_versions = json.loads((RUN/'logs/environment.json').read_text())['packages']
actual_versions = {name:metadata.version(name) for name in expected_versions}
assert actual_versions == expected_versions, 'Versions differ. Restart the runtime after installation and rerun.'
STAGE = RUN/'stage2'
STAGE.mkdir(exist_ok=True)
LOCAL = Path('/content/cindertrace_stage2')/RUN_ID
LOCAL.mkdir(parents=True,exist_ok=True)
print('Resumed:',RUN_ID)
print('Pilot months:',CONFIG['pilot_months'])

import numpy as np
import shutil,time,uuid,requests
from pyhdf.SD import SD,SDC
from pyproj import CRS,Transformer
from shapely import contains_xy

SELECTION_PATH=STAGE/'selected_science_files.json'
SELECTION_BYTES=SELECTION_PATH.read_bytes()
SELECTION=json.loads(SELECTION_BYTES)
assert SELECTION['run_id']==RUN_ID
assert SELECTION['source_query_log_sha256']==hashlib.sha256((RUN/'inventory/query_log.json').read_bytes()).hexdigest(), 'Stage 1 source inventory changed.'
BOUNDARY_PATH=STAGE/'Sri_Lanka_pilot_boundary.geojson'
assert hashlib.sha256(BOUNDARY_PATH.read_bytes()).hexdigest()==SELECTION['boundary_sha256']
REGION=shape(json.loads(BOUNDARY_PATH.read_text())['features'][0]['geometry'])
assert REGION.is_valid
RECORDS=SELECTION['files']
assert len(RECORDS)==60 and len({r['id'] for r in RECORDS})==60
for r in RECORDS:
    url=urllib.parse.urlsplit(r['download_url'])
    assert url.scheme=='https' and url.hostname=='data.lpdaac.earthdatacloud.nasa.gov'
    assert re.fullmatch(r'(MOD14A1|MYD14A1)\.A\d{7}\.h\d{2}v\d{2}\.061\.\d+',r['name'])

S3=RUN/'stage3';S3.mkdir(exist_ok=True)
RAW=RUN/'raw'/'verified_l3';RAW.mkdir(parents=True,exist_ok=True)
RECEIPTS=S3/'download_receipts';RECEIPTS.mkdir(exist_ok=True)
WORK=Path('/content/cindertrace_stage3')/RUN_ID;WORK.mkdir(parents=True,exist_ok=True)
first_receipt=json.loads((STAGE/'first_file_verification.json').read_text())
first_path=RAW/(first_receipt['name']+'.hdf')
assert first_path.exists() and hashlib.sha256(first_path.read_bytes()).hexdigest()==first_receipt['sha256'], 'Stage 2 verified file changed.'
print('Selection and boundary verified. Files:',len(RECORDS))
print('Existing verified sample:',first_path.name)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Resumed: 20261114T163518Z_5b1b1c54
Pilot months: {'2001-01': 'disputed', '2002-01': 'control', '2022-12': 'control', '2023-12': 'disputed'}
Selection and boundary verified. Files: 60
Existing verified sample: MOD14A1.A2001001.h25v08.061.2020058203517.hdf


## 3. Define file checks and resumable downloads

The cache is checked against receipts, catalogue byte sizes, HDF4 signatures, daily-grid dimensions and source dates. Only a fully verified download is copied to Drive. Retries report a filename and HTTP/error code without displaying tokens or signed redirect URLs. Previously verified files are reused.

The checks establish file identity and structure, not scientific observing completeness. Keep this notebook running until the final checkpoint. If a transfer fails, rerun the authentication and download cells; completed files remain cached.


In [14]:
class DownloadProblem(RuntimeError):
    """A deliberately nonsecret error message."""

def md_value(text,key):
    block=re.search(r'OBJECT\s*=\s*'+re.escape(key)+r'\b(.*?)END_OBJECT',text,re.S)
    assert block, f'Missing HDF metadata field: {key}'
    match=re.search(r'\bVALUE\s*=\s*"([^"]+)"',block.group(1))
    assert match, f'Invalid HDF metadata field: {key}'
    return match.group(1)

def file_check(path,record):
    assert path.exists(), 'Missing science file.'
    assert abs(path.stat().st_size-record['catalogue_size_bytes'])<=2, 'Catalogue/file byte-size mismatch.'
    with path.open('rb') as stream:
        assert stream.read(4)==b'\x0e\x03\x13\x01', 'Not an HDF4 science file.'
    h=SD(str(path),SDC.READ)
    try:
        names=h.datasets()
        assert 'FireMask' in names and 'QA' in names
        assert tuple(names['FireMask'][1])==(8,1200,1200)
        assert tuple(names['QA'][1])==(8,1200,1200)
        attrs=h.attributes();core=attrs.get('CoreMetadata.0','')
        start=md_value(core,'RANGEBEGINNINGDATE');end=md_value(core,'RANGEENDINGDATE')
        assert start==pd.Timestamp(record['time_start']).strftime('%Y-%m-%d')
        assert end==pd.Timestamp(record['time_end']).strftime('%Y-%m-%d')
        filename_day=re.search(r'\.A(\d{7})\.',record['name']).group(1)
        assert datetime.strptime(filename_day,'%Y%j').strftime('%Y-%m-%d')==start
        assert 'StructMetadata.0' in attrs
    finally:h.end()
    return {'id':record['id'],'name':record['name'],'product':record['product'],
        'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
        'time_start':record['time_start'],'time_end':record['time_end'],
        'selection_manifest_sha256':hashlib.sha256(SELECTION_BYTES).hexdigest()}

def cached_receipt(record):
    path=RAW/(record['name']+'.hdf')
    if not path.exists():return None
    receipt=file_check(path,record)
    receipt_path=RECEIPTS/(record['name']+'.json')
    if receipt_path.exists():
        prior=json.loads(receipt_path.read_text())
        assert prior['sha256']==receipt['sha256'] and prior['id']==record['id'], 'A cached file differs from its receipt.'
    else:receipt_path.write_text(json.dumps(receipt,indent=2))
    return receipt

def obtain_file(record,session):
    cached=cached_receipt(record)
    if cached is not None:return cached
    filename=record['name']+'.hdf';local=WORK/filename
    reason='unclassified transfer failure'
    for attempt in range(1,5):
        try:
            with session.get(record['download_url'],stream=True,timeout=(30,180),headers={'Cache-Control':'no-cache'}) as response:
                if response.status_code!=200:raise DownloadProblem(f'HTTP {response.status_code}')
                with local.open('wb') as stream:
                    for chunk in response.iter_content(128*1024):
                        if chunk:stream.write(chunk)
            receipt=file_check(local,record)
            temporary=RAW/(filename+'.partial')
            shutil.copy2(local,temporary)
            assert hashlib.sha256(temporary.read_bytes()).hexdigest()==receipt['sha256'], 'Drive copy checksum mismatch.'
            temporary.replace(RAW/filename)
            (RECEIPTS/(record['name']+'.json')).write_text(json.dumps(receipt,indent=2))
            local.unlink(missing_ok=True)
            return receipt
        except DownloadProblem as error:reason=str(error)
        except requests.exceptions.RequestException as error:reason=type(error).__name__
        except (AssertionError,OSError) as error:reason=type(error).__name__+' during file verification'
        local.unlink(missing_ok=True)
        (RAW/(filename+'.partial')).unlink(missing_ok=True)
        if attempt<4:
            print(f'Retrying {filename}: {reason}; attempt {attempt+1}/4',flush=True)
            time.sleep(min(2**attempt,8))
    raise DownloadProblem(f'{filename}: {reason}. Completed files are cached; rerun login and download cells to resume.')


## 4. Sign in and download the remaining verified files

Enter Earthdata credentials only in the prompts. This session does not persist them. The download session is used only with the NASA URLs verified above. A new Colab runtime needs a new login; completed Drive checkpoints survive disconnects.

Reference: https://earthaccess.readthedocs.io/en/stable/api/


In [15]:
import earthaccess
auth=earthaccess.login(strategy='interactive',persist=False)
assert auth.authenticated, 'Earthdata login did not succeed.'
http=earthaccess.get_requests_https_session()
receipts=[]
try:
    for i,record in enumerate(RECORDS,1):
        receipts.append(obtain_file(record,http))
        print(f'Verified {i}/{len(RECORDS)}: {record["name"]}',flush=True)
finally:http.close()
assert len(receipts)==len(RECORDS)
(S3/'download_manifest.json').write_text(json.dumps({'run_id':RUN_ID,'files':receipts},indent=2))
print('All 60 science files verified. Proceed to the processing checks.')


Verified 1/60: MOD14A1.A2000361.h26v08.061.2020058163429
Verified 2/60: MOD14A1.A2000361.h25v08.061.2020058162112
Verified 3/60: MOD14A1.A2001001.h25v08.061.2020058203517
Verified 4/60: MOD14A1.A2001001.h26v08.061.2020058203154
Verified 5/60: MOD14A1.A2001009.h25v08.061.2020061194747
Verified 6/60: MOD14A1.A2001009.h26v08.061.2020061194706
Verified 7/60: MOD14A1.A2001017.h25v08.061.2020061205406
Verified 8/60: MOD14A1.A2001017.h26v08.061.2020061205916
Verified 9/60: MOD14A1.A2001025.h25v08.061.2020061220531
Verified 10/60: MOD14A1.A2001025.h26v08.061.2020061220407
Verified 11/60: MOD14A1.A2001361.h26v08.061.2020067130501
Verified 12/60: MOD14A1.A2001361.h25v08.061.2020067130941
Verified 13/60: MOD14A1.A2002001.h25v08.061.2020067164920
Verified 14/60: MOD14A1.A2002001.h26v08.061.2020067164903
Verified 15/60: MOD14A1.A2002009.h25v08.061.2020067232738
Verified 16/60: MOD14A1.A2002009.h26v08.061.2020067234157
Verified 17/60: MOD14A1.A2002017.h26v08.061.2020068070731
Verified 18/60: MOD14A1

## 5. Define daily-layer accounting and check it with synthetic data

December and January packages can overlap. Each product/tile/day is counted once only after the complete fire-mask bytes, QA bytes and grid definition are compared. Identical overlap provenance is retained. Conflicting overlaps stop the calculation.

Native classes remain separate: missing input (0), unprocessed (1/2), water non-fire (3), cloud (4), land non-fire (5), unknown (6), and low/nominal/high fire (7/8/9). A usable classification is class 3, 5, 7, 8 or 9 with QA land/water bits other than 3 (missing). Its fraction is a diagnostic, not an automatic eligibility threshold. These daily composites do not enumerate every overpass.


In [16]:
CLASS_NAMES={0:'missing_input',1:'not_processed_1',2:'not_processed_2',3:'nonfire_water',
             4:'cloud',5:'nonfire_land',6:'unknown',7:'fire_low',8:'fire_nominal',9:'fire_high'}


def grid_definition(text,nrows,ncols):
    def pair(key):
        match=re.search(re.escape(key)+r'\s*=\s*\(\s*([-+0-9.eE]+)\s*,\s*([-+0-9.eE]+)\s*\)',text)
        assert match, f'Missing grid coordinate: {key}'
        return tuple(map(float,match.groups()))
    assert re.search(r'Projection\s*=\s*GCTP_SNSOID',text), 'Unexpected grid projection.'
    parameters=re.search(r'ProjParams\s*=\s*\(\s*([-+0-9.eE]+)',text)
    assert parameters and abs(float(parameters.group(1))-6371007.181)<.01, 'Unexpected MODIS sphere radius.'
    x0,y0=pair('UpperLeftPointMtrs');x1,y1=pair('LowerRightMtrs')
    dx=(x1-x0)/ncols;dy=(y1-y0)/nrows
    assert dx>0 and dy<0
    return (x0,y0,dx,dy,nrows,ncols)

def region_mask(grid):
    x0,y0,dx,dy,nrows,ncols=grid
    transform=Transformer.from_crs(CRS.from_proj4('+proj=sinu +R=6371007.181 +nadgrids=@null +wktext'),4326,always_xy=True)
    xx,yy=np.meshgrid(x0+(np.arange(ncols)+.5)*dx,y0+(np.arange(nrows)+.5)*dy)
    lon,lat=transform.transform(xx,yy)
    mask=contains_xy(REGION,lon,lat)
    assert mask.any(), 'A selected tile contains no regional pixel centres; review spatial selection.'
    return mask

def add_layer(rows,seen,exposure,fire,qa,mask,day,product,tile,granule,grid):
    key=(str(day.date()),product,tile)
    digest=hashlib.sha256(repr(grid).encode()+fire.tobytes()+qa.tobytes()).hexdigest()
    if key in seen:
        previous=rows[seen[key]]
        assert previous['daily_content_sha256']==digest, f'Conflicting daily layers: {key}'
        previous['equivalent_granule_ids']=sorted(set(previous['equivalent_granule_ids']+[granule]))
        return 0
    values=fire[mask]
    assert values.size and np.all((values>=0)&(values<=9)), 'Unexpected fire-mask class.'
    quality=qa[mask].astype(np.uint8)
    usable=np.isin(values,[3,5,7,8,9])&((quality&3)!=3)
    row={'date':day,'month':day.strftime('%Y-%m'),'product':product,'tile':tile,
         'region_pixels':int(mask.sum()),'daily_content_sha256':digest,'equivalent_granule_ids':[granule]}
    row.update({name:int((values==value).sum()) for value,name in CLASS_NAMES.items()})
    row['fire_pixel_days']=sum(row[name] for name in ['fire_low','fire_nominal','fire_high'])
    row['qa_missing']=int(((quality&3)==3).sum())
    row['usable_pixel_days']=int(usable.sum())
    seen[key]=len(rows);rows.append(row)
    exposure_key=(row['month'],product,tile)
    if exposure_key not in exposure:exposure[exposure_key]=np.zeros(len(values),dtype=np.uint16)
    exposure[exposure_key]+=usable.astype(np.uint16)
    return 1

def read_band(handle,name):
    band=handle.select(name)
    try:return band.get()
    finally:band.endaccess()


In [17]:
# Synthetic safety checks only; these are not satellite results.
test_rows=[];test_seen={};test_exposure={}
fake_fire=np.array([[0,4],[5,9]],dtype=np.uint8)
fake_qa=np.full((2,2),2,dtype=np.uint8);fake_mask=np.ones((2,2),dtype=bool)
test_day=pd.Timestamp('2001-01-01')
assert add_layer(test_rows,test_seen,test_exposure,fake_fire,fake_qa,fake_mask,test_day,'MOD14A1','test','a',('synthetic',))==1
assert add_layer(test_rows,test_seen,test_exposure,fake_fire,fake_qa,fake_mask,test_day,'MOD14A1','test','b',('synthetic',))==0
assert len(test_rows)==1 and test_rows[0]['fire_pixel_days']==1 and test_rows[0]['usable_pixel_days']==2
assert test_exposure[('2001-01','MOD14A1','test')].max()==1
changed=fake_fire.copy();changed[1,1]=8
try:
    add_layer(test_rows,test_seen,test_exposure,changed,fake_qa,fake_mask,test_day,'MOD14A1','test','c',('synthetic',))
except AssertionError:pass
else:raise AssertionError('Conflicting duplicate was not rejected.')
print('Synthetic checks passed: missing/cloud/fire separation, identical overlap deduplication, distinct-day exposure, conflict rejection.')


Synthetic checks passed: missing/cloud/fire separation, identical overlap deduplication, distinct-day exposure, conflict rejection.


## 6. Process the verified daily layers

This clips native grid-pixel centres to the frozen region polygon and retains only the pilot calendar days. Terra and Aqua remain separate. The class sums must account for every regional pixel, every expected tile/day must be present, and every pixel's usable-day count must be at most the number of calendar days.

Having all daily layers is different from having usable observations everywhere. Missing-input, cloud and unknown states remain visible in the results.


In [18]:
# Recheck cached files even when resuming directly at this cell.
checked=[cached_receipt(record) for record in RECORDS]
assert all(receipt is not None for receipt in checked), 'Complete the downloads before processing.'
rows=[];seen={};exposure={};masks={};tile_grids={};duplicates_removed=0
for i,record in enumerate(RECORDS,1):
    path=RAW/(record['name']+'.hdf')
    handle=SD(str(path),SDC.READ)
    try:
        fire=read_band(handle,'FireMask');qa=read_band(handle,'QA')
        assert fire.shape==qa.shape==(8,1200,1200)
        grid=grid_definition(handle.attributes()['StructMetadata.0'],*fire.shape[1:])
        tile=record['name'].split('.')[2]
        if tile in tile_grids:assert tile_grids[tile]==grid, 'Grid definition changed for a tile.'
        else:tile_grids[tile]=grid
        if tile not in masks:masks[tile]=region_mask(grid)
        mask=masks[tile]
        start=pd.Timestamp(record['time_start']).tz_convert('UTC').tz_localize(None).normalize()
        last=pd.Timestamp(record['time_end']).tz_convert('UTC').tz_localize(None)
        for offset in range(fire.shape[0]):
            day=start+pd.Timedelta(days=offset)
            if day>last or day.strftime('%Y-%m') not in CONFIG['pilot_months']:continue
            added=add_layer(rows,seen,exposure,fire[offset],qa[offset],mask,day,record['product'],tile,record['id'],grid)
            duplicates_removed+=1-added
    finally:handle.end()
    print(f'Read {i}/{len(RECORDS)}: {record["name"]}',flush=True)

DAILY_TILES=pd.DataFrame(rows)
assert not DAILY_TILES.duplicated(['date','product','tile']).any()
assert DAILY_TILES[list(CLASS_NAMES.values())].sum(axis=1).eq(DAILY_TILES.region_pixels).all()
missing_tile_days=[]
for source in SELECTION['summary']:
    if source['selected_files']==0:
        assert source['product']=='MYD14A1' and source['month'] in ['2001-01','2002-01'], 'Unexpected empty product/month.'
        continue
    tiles=source['tiles'].split(',')
    dates=pd.date_range(source['month']+'-01',periods=pd.Timestamp(source['month']+'-01').days_in_month)
    for day in dates:
        for tile in tiles:
            key=(str(day.date()),source['product'],tile)
            if key not in seen:missing_tile_days.append(key)
assert not missing_tile_days, f'Missing tile/day layers: {missing_tile_days[:10]}'
for (month,product,tile),counts in exposure.items():
    assert counts.max()<=pd.Timestamp(month+'-01').days_in_month, 'Exposure days were double counted.'
print('Unique tile/day rows:',len(DAILY_TILES))
print('Identical overlapping tile/day copies removed:',duplicates_removed)
print('All expected daily tile layers present. Pixel quality remains separately assessed below.')


Read 1/60: MOD14A1.A2000361.h26v08.061.2020058163429
Read 2/60: MOD14A1.A2000361.h25v08.061.2020058162112
Read 3/60: MOD14A1.A2001001.h25v08.061.2020058203517
Read 4/60: MOD14A1.A2001001.h26v08.061.2020058203154
Read 5/60: MOD14A1.A2001009.h25v08.061.2020061194747
Read 6/60: MOD14A1.A2001009.h26v08.061.2020061194706
Read 7/60: MOD14A1.A2001017.h25v08.061.2020061205406
Read 8/60: MOD14A1.A2001017.h26v08.061.2020061205916
Read 9/60: MOD14A1.A2001025.h25v08.061.2020061220531
Read 10/60: MOD14A1.A2001025.h26v08.061.2020061220407
Read 11/60: MOD14A1.A2001361.h26v08.061.2020067130501
Read 12/60: MOD14A1.A2001361.h25v08.061.2020067130941
Read 13/60: MOD14A1.A2002001.h25v08.061.2020067164920
Read 14/60: MOD14A1.A2002001.h26v08.061.2020067164903
Read 15/60: MOD14A1.A2002009.h25v08.061.2020067232738
Read 16/60: MOD14A1.A2002009.h26v08.061.2020067234157
Read 17/60: MOD14A1.A2002017.h26v08.061.2020068070731
Read 18/60: MOD14A1.A2002017.h25v08.061.2020068071005
Read 19/60: MOD14A1.A2002025.h26v08.0

## 7. Save and inspect the pilot results

The output unit is **fire-active 1 km grid-pixel-days per satellite**, not raw detections or production H3 cell-days. Do not add Terra and Aqua values and label them deduplicated combined MODIS activity.

`usable_pixel_day_fraction` includes non-fire water/land and fire classes with nonmissing QA. It describes this frozen geographic mask. It is not a land-only clear-sky fraction, an observation probability or a physical no-fire claim. The spatial-support table reports locations that never had a usable daily classification and the distribution of usable days.


In [19]:
numeric=['region_pixels',*CLASS_NAMES.values(),'fire_pixel_days','qa_missing','usable_pixel_days']
DAILY=DAILY_TILES.groupby(['date','month','product'])[numeric].sum().reset_index()
MONTHLY=DAILY.groupby(['month','product'])[numeric].sum().reset_index()
MONTHLY['role']=MONTHLY.month.map(CONFIG['pilot_months'])
MONTHLY['usable_pixel_day_fraction']=MONTHLY.usable_pixel_days/MONTHLY.region_pixels
MONTHLY['cloud_pixel_day_fraction']=MONTHLY.cloud/MONTHLY.region_pixels
MONTHLY['unknown_pixel_day_fraction']=MONTHLY.unknown/MONTHLY.region_pixels
MONTHLY['days_with_layers']=MONTHLY.apply(lambda r:int(DAILY[(DAILY.month==r['month'])&(DAILY['product']==r['product'])].date.nunique()),axis=1)
MONTHLY['calendar_days']=MONTHLY.month.map(lambda m:pd.Timestamp(m+'-01').days_in_month)
assert MONTHLY.days_with_layers.eq(MONTHLY.calendar_days).all()
support=[]
for month,product in sorted({(m,p) for m,p,t in exposure}):
    counts=np.concatenate([v for (m,p,t),v in exposure.items() if m==month and p==product])
    support.append({'month':month,'product':product,'region_pixels':len(counts),
        'pixels_never_usable':int((counts==0).sum()),'fraction_never_usable':float((counts==0).mean()),
        'p10_usable_days':float(np.quantile(counts,.1)),'median_usable_days':float(np.median(counts)),
        'p90_usable_days':float(np.quantile(counts,.9))})
SUPPORT=pd.DataFrame(support)

ANALYSIS=S3/'analyses'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
ANALYSIS.mkdir(parents=True,exist_ok=False)
DAILY_TILES.to_parquet(ANALYSIS/'daily_tile_counts.parquet',index=False)
DAILY.to_parquet(ANALYSIS/'daily_regional_counts.parquet',index=False)
MONTHLY.to_csv(ANALYSIS/'monthly_screening.csv',index=False)
SUPPORT.to_csv(ANALYSIS/'spatial_observation_support.csv',index=False)
report={'run_id':RUN_ID,'stage':'daily_L3_observation_screening','unit':'fire-active 1km grid-pixel-days per satellite',
    'files_verified':len(checked),'unique_tile_day_rows':len(DAILY_TILES),
    'identical_overlap_copies_removed':duplicates_removed,'missing_tile_day_layers':missing_tile_days,
    'monthly_screening':MONTHLY.to_dict('records'),'spatial_observation_support':support,
    'selection_manifest_sha256':hashlib.sha256(SELECTION_BYTES).hexdigest(),
    'boundary_sha256':SELECTION['boundary_sha256'],'environment':actual_versions,
    'method_sha256':'41ad98261e630392449361179b45191e443d115a9bbe3c91a0169e7bbf599077','file_receipts':checked,
    'zero_restoration_enabled':False,'production_export_enabled':False,
    'limits':['Daily composites do not enumerate all overpasses or prove swath/archive completeness.',
              'Zero fire pixels describe observed daily-product activity; obscured or short-lived fires can be missed.',
              'Terra/Aqua are reported separately and have not been deduplicated into combined H3 cell-days.',
              'Thresholds for comparison eligibility and replacement/allocation methods remain unvalidated.']}
(ANALYSIS/'report.json').write_text(json.dumps(report,indent=2,allow_nan=False))
(S3/'latest_analysis.json').write_text(json.dumps({'path':str(ANALYSIS.relative_to(RUN)),
    'report_sha256':hashlib.sha256((ANALYSIS/'report.json').read_bytes()).hexdigest()},indent=2))
print('ANALYSIS:',ANALYSIS)
display(MONTHLY[['month','role','product','fire_pixel_days','usable_pixel_day_fraction','cloud_pixel_day_fraction','missing_input','unknown','days_with_layers']])
display(SUPPORT)
print('All screening results saved. Zero restoration and production export remain disabled.')


ANALYSIS: /content/drive/MyDrive/cindertrace_clean_rebuild/runs/20261114T163518Z_5b1b1c54/stage3/analyses/20261114T171725Z_9173a5c9


,month,role,product,fire_pixel_days,usable_pixel_day_fraction,cloud_pixel_day_fraction,missing_input,unknown,days_with_layers
0,2001-01,disputed,MOD14A1,0,0.630214,0.369783,0,8,31
1,2002-01,control,MOD14A1,9,0.816156,0.183827,16,25,31
2,2022-12,control,MOD14A1,0,0.594622,0.405366,12,17,31
3,2022-12,control,MYD14A1,4,0.565946,0.391558,101832,5,31
4,2023-12,disputed,MOD14A1,0,0.659352,0.340418,550,0,31
5,2023-12,disputed,MYD14A1,0,0.565652,0.369481,155447,3,31


,month,product,region_pixels,pixels_never_usable,fraction_never_usable,p10_usable_days,median_usable_days,p90_usable_days
0,2001-01,MOD14A1,77304,0,0.0,17.0,20.0,22.0
1,2002-01,MOD14A1,77304,0,0.0,23.0,26.0,28.0
2,2022-12,MOD14A1,77304,0,0.0,16.0,18.0,21.0
3,2022-12,MYD14A1,77304,0,0.0,15.0,18.0,20.0
4,2023-12,MOD14A1,77304,0,0.0,17.0,21.0,23.0
5,2023-12,MYD14A1,77304,0,0.0,14.0,17.0,21.0


All screening results saved. Zero restoration and production export remain disabled.


## Stage 3 checkpoint

Save the executed notebook back to the project and share its outputs, as you did for Stages 1 and 2. The important outputs are:

1. `All 60 science files verified`.
2. Unique tile/day rows and the identical-overlap count (the reference inventory is expected to produce 372 unique tile/day rows; review any difference).
3. The monthly screening table and spatial-observation-support table.
4. The printed `ANALYSIS` path.

If a cell stops, share the cell number and the nonsecret error message. Rerun the authentication/download cells after a disconnected session; cached verified files are reused. Do not rerun processing past a failed conflict or completeness check.

Next we will reconcile actual swath observations and native detections, including direct-archive recovery of catalogue omissions. Only after that evidence is reviewed will we define zero-month eligibility and rebuild H3/category/confidence tables. This stage is a screening checkpoint, not certification of items 1–7.
